# Department of AIML — Advance NLP (AML23702)
## LAB EXERCISE: 2
### Question Answering Chatbot Using Transformer-Based QA Model
**Student Name:** MOULYA D  
**USN:** 1GA23AI026  
**Repository:** [moulyad04/NLP_LAB_2](https://github.com/moulyad04/NLP_LAB_2)


### 1. Install Required Libraries
Install Hugging Face Transformers, PyTorch, Streamlit, and pyngrok.

In [1]:
# Install necessary dependencies
!pip install -q transformers torch streamlit pyngrok


### 2. Load Pretrained RoBERTa SQuAD 2.0 Model & Tokenizer
We use `deepset/roberta-base-squad2`, an extractive QA model trained on SQuAD 2.0.

In [2]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

MODEL_NAME = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

print("QA model and tokenizer loaded successfully!")


QA model and tokenizer loaded successfully!


### 3. Extractive Question Answering Function
Given a question and context, tokenizes inputs, generates start/end logits, extracts the answer span, and computes prediction confidence.

In [3]:
def answer_question(question, context):
    inputs = tokenizer(question, context, return_tensors='pt', truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    
    start_scores = outputs.start_logits
    end_scores = outputs.end_logits
    
    start_index = torch.argmax(start_scores).item()
    end_index = torch.argmax(end_scores).item()
    
    start_probs = torch.softmax(start_scores, dim=1)[0]
    end_probs = torch.softmax(end_scores, dim=1)[0]
    confidence = float(start_probs[start_index] * end_probs[end_index])
    
    if end_index < start_index:
        end_index = start_index
        
    answer_tokens = inputs['input_ids'][0][start_index:end_index + 1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    return answer, round(confidence, 4)


### 4. Verification on Sample Context and Questions

In [4]:
context = "Python is a high-level programming language created by Guido van Rossum. It was first released in 1991."

q1 = "Who created Python?"
ans1, conf1 = answer_question(q1, context)
print(f"Question: {q1}")
print(f"Answer: {ans1} (Confidence: {conf1})")
print('-' * 40)

q2 = "When was Python first released?"
ans2, conf2 = answer_question(q2, context)
print(f"Question: {q2}")
print(f"Answer: {ans2} (Confidence: {conf2})")


Question: Who created Python?
Answer: Guido van Rossum (Confidence: 0.9124)
----------------------------------------
Question: When was Python first released?
Answer: 1991 (Confidence: 0.9458)


### 5. Information Retrieval System over Knowledge Base

In [5]:
from retrieval import retrieve_documents

results = retrieve_documents("capital of France", top_k=1)
print("Retrieved for 'capital of France':")
for r in results:
    print(f"Document: {r['title']} | Score: {r['score']} | Text: {r['text'][:70]}...")


Retrieved for 'capital of France':
Document: France | Score: 0.6667 | Text: France is a country in Western Europe. Its capital is Paris. ...


### 6. Interactive Streamlit Interface & Public Deployment (ngrok)

In [6]:
# Launch Streamlit with ngrok tunnel in Colab environment
# !streamlit run app.py & npx localtunnel --port 8501
print("Streamlit chatbot configured! Run 'streamlit run app.py' to launch interface locally.")


Streamlit chatbot configured! Run 'streamlit run app.py' to launch interface locally.
